In [ ]:
%pip install jikanpy-v4

In [ ]:
import time
import os
import json
import pandas as pd
from jikanpy import Jikan
from requests.exceptions import HTTPError
from jikanpy.exceptions import APIException

In [ ]:
# Kaggle working directory is /kaggle/working
import os
print('Working directory:', os.getcwd())
print('Output will be saved to /kaggle/working/')

In [ ]:
PROJECT_DIR = '/kaggle/working'
DATA_DIR = f'{PROJECT_DIR}/data'
SCRIPTS_DIR = f'{PROJECT_DIR}/scripts'

os.makedirs(DATA_DIR, exist_ok=True)
os.makedirs(SCRIPTS_DIR, exist_ok=True)
print(f'Data directory: {DATA_DIR}')

In [ ]:
OUTPUT_FILE = f'{DATA_DIR}/anime_full_dataset.csv'
STATE_FILE = f'{DATA_DIR}/crawler_state.json'

In [ ]:
jikan=Jikan()

In [ ]:
anime=jikan.anime(1)

In [ ]:
from jikanpy import Jikan
from jikanpy.exceptions import APIException
import pandas as pd
import time
import json
import os

def is_valid_anime(anime_data):
    data = anime_data.get('data', anime_data)

    #if data.get("rating") == "Rx - Hentai":
        #return False
    #if data.get("type") == "Music":
        #return False
    #if not data.get("synopsis"):
        #return False
    return True


def collect_anime_by_id(
    start_id=1,
    end_id=10000,
    output_file=OUTPUT_FILE,
    state_file=STATE_FILE,
    batch_size=50,
    save_interval=300
):
    jikan = Jikan()
    collected_batch = []
    backoff = 2

    stats = {
        'total_checked': 0,
        'valid_anime': 0,
        'not_found': 0,
        'filtered_out': 0,
        'errors': 0,
        'last_id': start_id
    }

    if os.path.exists(state_file):
        with open(state_file, "r") as f:
            state = json.load(f)
            start_id = state.get("last_id", start_id)
            stats = state.get("stats", stats)
            print(f"Resuming from ID {start_id}")
            print(f"Previous stats: {stats}")

    file_exists = os.path.isfile(output_file)

    print("\n" + "=" * 70)
    print(f"ANIME CRAWLER - KAGGLE MODE")
    print("=" * 70)
    print(f"ID Range: {start_id:,} to {end_id:,} ({end_id - start_id + 1:,} IDs)")
    print(f"Output: {output_file}")
    print(f"Auto-save: Every {batch_size} anime OR every {save_interval}s")
    print("-" * 70)

    start_time = time.time()
    last_save_time = time.time()

    for mal_id in range(start_id, end_id + 1):
        stats['total_checked'] += 1
        stats['last_id'] = mal_id

        try:
            response = jikan.anime(mal_id)
            anime = response.get('data', response)

            if not anime:
                stats['not_found'] += 1
                continue

            if not is_valid_anime(response):
                stats['filtered_out'] += 1
                continue

            anime_data = {
                "anime_id": anime.get("mal_id"),
                "title": anime.get("title"),
                "english_title": anime.get("title_english"),
                "japanese_title": anime.get("title_japanese"),
                "synopsis": anime.get("synopsis"),

                "type": anime.get("type"),
                "source": anime.get("source"),
                "episodes": anime.get("episodes"),
                "status": anime.get("status"),
                "duration": anime.get("duration"),
                "rating": anime.get("rating"),

                "year": anime.get("year"),
                "season": anime.get("season"),

                "score": anime.get("score"),
                "scored_by": anime.get("scored_by"),
                "members": anime.get("members"),
                "favorites": anime.get("favorites"),
                "rank": anime.get("rank"),
                "popularity": anime.get("popularity"),

                "genres": json.dumps([g["name"] for g in anime.get("genres", [])]),
                "explicit_genres": json.dumps([g["name"] for g in anime.get("explicit_genres", [])]),
                "themes": json.dumps([t["name"] for t in anime.get("themes", [])]),
                "demographics": json.dumps([d["name"] for d in anime.get("demographics", [])]),

                "studios": json.dumps([s["name"] for s in anime.get("studios", [])]),
            }

            collected_batch.append(anime_data)
            stats['valid_anime'] += 1
            backoff = 2

            if stats['valid_anime'] % 25 == 0:
                elapsed = time.time() - start_time
                rate = stats['total_checked'] / elapsed if elapsed > 0 else 0
                remaining = (end_id - mal_id) / rate if rate > 0 else 0

                print(f"[{stats['valid_anime']:4d}] ID {mal_id:5d} | {rate:.1f} IDs/s | ETA: {remaining/60:.0f}m")

            should_save_batch = len(collected_batch) >= batch_size
            should_save_time = (time.time() - last_save_time) >= save_interval

            if should_save_batch or should_save_time:
                if collected_batch:
                    df_batch = pd.DataFrame(collected_batch)
                    df_batch.to_csv(
                        output_file,
                        mode="a",
                        header=not file_exists,
                        index=False,
                        quoting=1
                    )
                    file_exists = True

                    elapsed_total = time.time() - start_time
                    print(f"\n{'='*70}")
                    print(f"CHECKPOINT | Saved {len(collected_batch)} anime to /kaggle/working/data/")
                    print(f"Total: {stats['valid_anime']} valid | {stats['not_found']} not found | {stats['filtered_out']} filtered")
                    print(f"Runtime: {elapsed_total/60:.1f}m | Avg: {stats['total_checked']/elapsed_total:.1f} IDs/s")
                    print(f"{'='*70}\n")

                    collected_batch = []

                with open(state_file, "w") as f:
                    json.dump({
                        "last_id": mal_id + 1,
                        "stats": stats,
                        "timestamp": time.strftime("%Y-%m-%d %H:%M:%S")
                    }, f, indent=2)

                last_save_time = time.time()

        except APIException as e:
            if e.status_code == 404:
                stats['not_found'] += 1
                continue
            elif e.status_code == 429:
                print(f"Rate limit at ID {mal_id}. Sleeping {backoff}s...")
                time.sleep(backoff)
                backoff = min(backoff * 2, 60)
                continue
            else:
                stats['errors'] += 1
                print(f"API Error {e.status_code} at ID {mal_id}: {str(e)[:50]}")
                time.sleep(3)
                continue

        except Exception as e:
            stats['errors'] += 1
            print(f"Error at ID {mal_id}: {str(e)[:50]}")
            time.sleep(5)
            continue

        time.sleep(0.7)

    if collected_batch:
        df_batch = pd.DataFrame(collected_batch)
        df_batch.to_csv(output_file, mode="a", header=not file_exists, index=False, quoting=1)
        print(f"\nFinal save: {len(collected_batch)} anime")

    with open(state_file, "w") as f:
        json.dump({
            "completed": True,
            "last_id": mal_id,
            "stats": stats,
            "timestamp": time.strftime("%Y-%m-%d %H:%M:%S")
        }, f, indent=2)

    total_time = time.time() - start_time
    print("\n" + "=" * 70)
    print("SESSION COMPLETE")
    print("=" * 70)
    print(f"Valid anime collected: {stats['valid_anime']:,}")
    print(f"Last ID processed: {stats['last_id']:,}")
    print(f"Not found (404): {stats['not_found']:,}")
    print(f"Filtered out: {stats['filtered_out']:,}")
    print(f"Errors: {stats['errors']:,}")
    print(f"Success rate: {100 * stats['valid_anime'] / stats['total_checked']:.1f}%")
    print(f"\nTotal time: {total_time/60:.1f} minutes ({total_time/3600:.2f} hours)")
    print(f"Average rate: {stats['total_checked']/total_time:.2f} IDs/second")
    print(f"\nAll data saved to /kaggle/working/data/")
    print("=" * 70)

    return stats

print("Collection script loaded and ready")
print(f"Output location: {OUTPUT_FILE}")

In [ ]:
def get_last_saved_id(filename):
    if not os.path.exists(filename):
        return 1

    df = pd.read_csv(filename, usecols=["anime_id"])
    return df["anime_id"].max() + 1

## ⚠️ Important: Kaggle Session Limits
Kaggle sessions last **max 9 hours** and `/kaggle/working` is wiped when the session ends.
- The crawler saves progress to `crawler_state.json` so it can resume mid-run
- **Before your session ends**, run the download cell at the bottom to save your CSV
- On the next session, re-upload your CSV and state file, then re-run — the crawler will resume from where it left off

In [ ]:
time.sleep(3)

stats = collect_anime_by_id(
    start_id=get_last_saved_id(OUTPUT_FILE),
    end_id=80000,
    batch_size=50,
    save_interval=300
)

## Resuming in a New Session
1. Start a new Kaggle notebook
2. Upload your saved `anime_full_dataset.csv` and `crawler_state.json` to `/kaggle/working/data/`
3. Run all cells — the crawler will automatically resume from the last saved ID

In [ ]:
# Run this cell to download your progress before the session ends
from kaggle_secrets import UserSecretsClient
from IPython.display import FileLink
import os

print('Files available for download:')
for f in [OUTPUT_FILE, STATE_FILE]:
    if os.path.exists(f):
        size = os.path.getsize(f) / (1024*1024)
        print(f'  {f} ({size:.1f} MB)')
        display(FileLink(f))
    else:
        print(f'  {f} - not found')